# Trustworthiness and Selective Prediction Benchmark

## Paper Narrative

The main story here is that **even if prediction markets are informative on average, they are not equally trustworthy in every market state**.

This benchmark answers the core paper question:
- when should we trust the current market probability,
- when is it better to abstain,
- and can we identify reliable states better than simple confidence-based heuristics.

For the paper, this is the uncertainty and selective-prediction layer on top of the forecasting benchmark: it shifts the story from “what is the probability?” to “when is that probability worth trusting?”

## Everyday Intuition: Why This Might Work

The intuition is simple:
- even a strong market can be unreliable when liquidity is thin or information has not yet fully propagated;
- the same quoted probability can mean very different things depending on volatility, activity, and path stability;
- a separate trust model can learn not only `what is likely`, but also `when this probability is reliable enough to act on`.

## What Data We Will Use

- market snapshots at a fixed horizon before resolution;
- the history of `yes_probability` up to that point;
- liquidity, activity, and instability features;
- when available: coherence, instability, and manipulation proxies.

## What Metrics To Track

- selective prediction quality on the covered subset;
- `mean log loss` after abstention;
- coverage curves;
- stability of the trust signal across folds.

## What Models To Train

- confidence margin as a simple heuristic baseline;
- a rule-based trust score;
- a learned regressor / ranker for trust risk;
- comparison of learned selective policies against simple heuristics.

## How To Read This Notebook

This notebook is organized as a **research tutorial for a trustworthy benchmark**.

The reading logic is:
1. We define what it means to “trust the market.”
2. We construct a dataset at a fixed horizon before resolution.
3. We build a trust target and show which components it contains.
4. We compare simple selective policies and learned baselines.
5. We inspect at which coverage levels quality actually improves.

## Task

**What we predict:**
- not the outcome directly, but **how much the current market probability should be trusted**.

**Input:**
- the market state at a fixed horizon before resolution;
- the probability dynamics up to that point;
- liquidity, activity, and instability features;
- when available: coherence / manipulation / external-evidence proxies.

**Target:**
- the error of the market probability;
- future path instability;
- a composite risk score for selective prediction.

**Why this task matters:**
- a prediction market can be informative, but not equally trustworthy in every state;
- for NeurIPS this is already an uncertainty-aware and selective-prediction story, not just forecasting.

In [1]:
# If needed in a fresh notebook environment:
# %pip install -r ../requirements.txt

from __future__ import annotations

import ast
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "benchmarks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "benchmarks"))

from benchmark_utils import (
    DEFAULT_DB_PATH,
    add_time_features,
    build_multi_horizon_terminal_dataset,
    build_repricing_dataset,
    connect,
    load_eligible_markets,
    load_probabilities_for_markets,
    rolling_time_splits,
)
from covariate_utils import (
    add_lagged_covariate_features,
    asof_join_covariates,
    load_external_covariates,
    pivot_covariates_to_wide,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DB_PATH = DEFAULT_DB_PATH
DOMAINS = ("geopolitics", "finance_economy")
EXTERNAL_COVARIATES_PATH = REPO_ROOT / "data" / "external_covariates"
TAG_RE = re.compile(r"[A-Za-z][A-Za-z0-9_+-]+")
EVENT_PATTERNS = {
    "election": r"election|president|prime minister|mayor|governor|parliament",
    "military": r"strike|war|missile|ceasefire|attack|troops|nuclear",
    "policy": r"tariff|fed|rate|ban|approval|regulation|sanction|etf",
    "corporate": r"earnings|revenue|ipo|acquisition|merger|bankruptcy",
    "crypto": r"bitcoin|btc|ethereum|eth|solana|crypto|token|airdrop",
}
CANDIDATE_PATTERNS = (
    re.compile(r"^will .+ be elected the next president of (.+)$"),
    re.compile(r"^will .+ win the (.+) election$"),
    re.compile(r"^which party will win (.+)$"),
)


def parse_listish(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return [chunk.strip() for chunk in text.split(",") if chunk.strip()]


def slug_family(slug: object) -> str:
    text = str(slug or "").lower()
    text = re.sub(r"-\d+(?:-\d+)+$", "", text)
    text = re.split(r"-(?:by|before|after|on|during|in)-", text, maxsplit=1)[0]
    text = re.sub(r"-(?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december).*", "", text)
    text = re.sub(r"-+$", "", text)
    return text or "unknown_family"


def candidate_family(question: object) -> str:
    q = re.sub(r"\s+", " ", str(question or "").lower()).strip(" ?")
    for pattern in CANDIDATE_PATTERNS:
        match = pattern.match(q)
        if match:
            return match.group(1).strip()
    return ""


def build_market_text(markets_df: pd.DataFrame) -> pd.DataFrame:
    work = markets_df.copy()
    work["tag_list"] = work["tag_labels"].apply(parse_listish)
    matched_domains = work["matched_domains"] if "matched_domains" in work.columns else pd.Series("", index=work.index)
    work["matched_domain_list"] = matched_domains.apply(parse_listish)
    work["question"] = work["question"].fillna("")
    work["description"] = work["description"].fillna("")
    work["resolution_source"] = work["resolution_source"].fillna("")
    work["market_text"] = (
        work["question"]
        + " [SEP] "
        + work["description"]
        + " [SEP] tags: "
        + work["tag_list"].apply(lambda values: " ".join(values))
        + " [SEP] source: "
        + work["resolution_source"]
    )
    work["family_id"] = work["market_slug"].apply(slug_family)
    work["candidate_family_id"] = work["question"].apply(candidate_family)
    work["tag_count"] = work["tag_list"].apply(len)
    work["matched_domain_count"] = work["matched_domain_list"].apply(len)
    work["question_char_len"] = work["question"].str.len()
    work["description_char_len"] = work["description"].str.len()
    work["has_resolution_source"] = work["resolution_source"].ne("").astype(int)
    for keyword, pattern in EVENT_PATTERNS.items():
        work[f"kw_{keyword}"] = work["market_text"].str.contains(pattern, case=False, regex=True).astype(int)

    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=2500)
    tfidf = vectorizer.fit_transform(work["market_text"])
    if tfidf.shape[1] >= 2 and len(work) >= 3:
        n_components = int(min(16, len(work) - 1, tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        embeddings = svd.fit_transform(tfidf)
        for idx in range(embeddings.shape[1]):
            work[f"text_svd_{idx:02d}"] = embeddings[:, idx]
        similarity = cosine_similarity(embeddings)
        np.fill_diagonal(similarity, -1.0)
        work["duplicate_neighbor_count"] = (similarity >= 0.82).sum(axis=1)
        work["max_text_similarity"] = np.where(len(work) > 1, similarity.max(axis=1), 0.0)
    else:
        work["duplicate_neighbor_count"] = 0
        work["max_text_similarity"] = 0.0

    family_stats = (
        work.groupby("family_id", dropna=False)
        .agg(
            family_market_count=("market_id", "size"),
            family_volume_sum=("volume_num", "sum"),
            family_end_date_span_days=("end_date", lambda s: (s.max() - s.min()).total_seconds() / 86400.0 if len(s) > 1 else 0.0),
        )
        .reset_index()
    )
    candidate_stats = (
        work.loc[work["candidate_family_id"].ne("")]
        .groupby("candidate_family_id", dropna=False)
        .agg(candidate_market_count=("market_id", "size"), candidate_volume_sum=("volume_num", "sum"))
        .reset_index()
    )
    work = work.merge(family_stats, on="family_id", how="left")
    work = work.merge(candidate_stats, on="candidate_family_id", how="left")
    work["candidate_market_count"] = work["candidate_market_count"].fillna(0)
    work["candidate_volume_sum"] = work["candidate_volume_sum"].fillna(0.0)
    return work


def load_multi_domain_markets(conn, *, domains, max_markets_per_domain, min_probability_rows):
    frames = []
    for domain in domains:
        frame = load_eligible_markets(
            conn,
            domain=domain,
            max_markets=max_markets_per_domain,
            min_probability_rows=min_probability_rows,
        )
        frames.append(frame)
    markets_df = pd.concat(frames, ignore_index=True)
    markets_df = markets_df.sort_values(["primary_domain", "volume_num", "created_at"], ascending=[True, False, False], kind="stable")
    markets_df = markets_df.drop_duplicates(subset=["market_id"], keep="first").reset_index(drop=True)
    return build_market_text(markets_df)


def load_optional_covariates(path: Path):
    if not path.exists():
        return None
    covariates = load_external_covariates(path)
    if covariates.empty:
        return None
    value_col = "close" if "close" in covariates.columns else "value"
    wide = pivot_covariates_to_wide(covariates, value_col=value_col)
    features = add_lagged_covariate_features(wide, lags=(1, 12, 288), pct_change=True)
    return features


def join_covariates(base_df: pd.DataFrame, covariate_df: pd.DataFrame | None, *, time_col: str) -> pd.DataFrame:
    if covariate_df is None or covariate_df.empty:
        return base_df
    return asof_join_covariates(base_df, covariate_df, base_time_col=time_col, max_age="7D")


def add_manipulation_proxies(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6
    if "lookback_24h_trade_count_sum" in out.columns:
        trade_count = out["lookback_24h_trade_count_sum"].fillna(0.0)
        total_size = out["lookback_24h_total_size_sum"].fillna(0.0)
        volatility = out["lookback_24h_volatility"].fillna(0.0)
        abs_move = out["lookback_24h_abs_move_mean"].fillna(0.0)
        max_move = out["lookback_24h_abs_move_max"].fillna(0.0)
        directional = out["lookback_24h_yes_probability_change"].fillna(0.0)
        observed_share = out["lookback_24h_observed_trade_share"].fillna(0.0)
        staleness = out.get("snapshot_staleness_hours", pd.Series(0.0, index=out.index)).fillna(0.0)
    else:
        trade_count = out["trade_count_sum"].fillna(0.0)
        total_size = out["total_size_sum"].fillna(0.0)
        volatility = out["recent_volatility"].fillna(0.0)
        abs_move = out["recent_abs_move_mean"].fillna(0.0)
        max_move = out["recent_abs_move_max"].fillna(0.0)
        directional = out["recent_directional_move"].fillna(0.0)
        observed_share = out["observed_trade_share"].fillna(0.0)
        staleness = pd.Series(0.0, index=out.index)

    confidence = out["confidence_margin"].fillna(0.0)
    out["avg_trade_size_proxy"] = total_size / (trade_count + eps)
    out["move_per_trade_proxy"] = directional.abs() / (trade_count + eps)
    out["move_per_dollar_proxy"] = directional.abs() / (total_size + eps)
    out["burstiness_proxy"] = max_move / (abs_move + eps)
    out["wash_proxy"] = observed_share * trade_count / (directional.abs() + volatility + 1e-4)
    out["conviction_without_flow_proxy"] = confidence / (observed_share + 0.05)
    out["stale_conviction_proxy"] = confidence * (1.0 + staleness)
    out["volatility_gap_proxy"] = volatility / (observed_share + 0.05)
    return out


def latest_probability_before(panel: pd.DataFrame, cutoff: pd.Timestamp) -> float:
    history = panel.loc[panel["timestamp_utc"] <= cutoff]
    if history.empty:
        return float("nan")
    return float(history["yes_probability"].iloc[-1])


def attach_family_snapshot_features(dataset: pd.DataFrame, probabilities_df: pd.DataFrame, markets_df: pd.DataFrame, *, time_col: str, prob_col: str) -> pd.DataFrame:
    panels = {market_id: frame.reset_index(drop=True) for market_id, frame in probabilities_df.groupby("market_id", sort=False)}
    family_members = markets_df.groupby("family_id", dropna=False)["market_id"].agg(list).to_dict()
    candidate_members = (
        markets_df.loc[markets_df["candidate_family_id"].ne("")]
        .groupby("candidate_family_id", dropna=False)["market_id"].agg(list).to_dict()
    )
    family_map = markets_df.set_index("market_id")["family_id"].to_dict()
    candidate_map = markets_df.set_index("market_id")["candidate_family_id"].to_dict()
    end_date_map = markets_df.set_index("market_id")["end_date"].to_dict()

    rows = []
    for row in dataset[["market_id", time_col, prob_col]].itertuples(index=False):
        market_id = str(row.market_id)
        cutoff = pd.Timestamp(getattr(row, time_col))
        current_prob = float(getattr(row, prob_col))
        family_id = family_map.get(market_id, "unknown_family")
        sibling_probs = []
        earlier_probs = []
        later_probs = []
        for sibling_id in family_members.get(family_id, []):
            if sibling_id == market_id:
                continue
            sibling_prob = latest_probability_before(panels[sibling_id], cutoff) if sibling_id in panels else float("nan")
            if np.isnan(sibling_prob):
                continue
            sibling_probs.append(sibling_prob)
            if end_date_map.get(sibling_id) < end_date_map.get(market_id):
                earlier_probs.append(sibling_prob)
            elif end_date_map.get(sibling_id) > end_date_map.get(market_id):
                later_probs.append(sibling_prob)

        candidate_family_id = candidate_map.get(market_id, "")
        candidate_probs = []
        if candidate_family_id:
            for sibling_id in candidate_members.get(candidate_family_id, []):
                sibling_prob = latest_probability_before(panels[sibling_id], cutoff) if sibling_id in panels else float("nan")
                if not np.isnan(sibling_prob):
                    candidate_probs.append(sibling_prob)

        later_violation = max(0.0, current_prob - min(later_probs)) if later_probs else 0.0
        earlier_violation = max(0.0, max(earlier_probs) - current_prob) if earlier_probs else 0.0
        candidate_sum = float(np.sum(candidate_probs)) if candidate_probs else current_prob
        rows.append(
            {
                "market_id": market_id,
                time_col: cutoff,
                "family_snapshot_size": len(sibling_probs) + 1,
                "family_prob_mean": float(np.mean(sibling_probs)) if sibling_probs else current_prob,
                "family_prob_std": float(np.std(sibling_probs)) if sibling_probs else 0.0,
                "family_prob_gap": current_prob - (float(np.mean(sibling_probs)) if sibling_probs else current_prob),
                "family_coherence_gap": later_violation + earlier_violation,
                "family_future_monotone_violation": later_violation,
                "family_past_monotone_violation": earlier_violation,
                "candidate_prob_sum_gap": abs(candidate_sum - 1.0) if candidate_probs else 0.0,
            }
        )

    features = pd.DataFrame(rows)
    return dataset.merge(features, on=["market_id", time_col], how="left")


def add_domain_dummies(df: pd.DataFrame) -> pd.DataFrame:
    if "primary_domain" not in df.columns:
        return df
    dummies = pd.get_dummies(df["primary_domain"], prefix="domain", dtype=float)
    return pd.concat([df, dummies], axis=1)


def safe_auc(y_true, pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, pred)


def clipped(values, eps=1e-6):
    return np.clip(np.asarray(values, dtype=float), eps, 1.0 - eps)


def summarize_by_group(df: pd.DataFrame, *, group_cols, metric_cols):
    return (
        df.groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )


## Dataset Construction

Here we build a trustworthiness benchmark at a fixed horizon before resolution.

Data sources:
- `markets` and `added_markets` to select eligible markets;
- `probabilities` for market probability history;
- derived features that estimate future instability and coherence pressure.

Filtering:
- we only keep markets with sufficient history;
- we use a snapshot close to the desired horizon;
- we exclude rows where the future is no longer meaningfully available or the target cannot be computed reliably.

**Unit of evaluation:**
- one row = one market at a fixed trust horizon.

In [2]:
MAX_MARKETS_PER_DOMAIN = 180
TARGET_HORIZON_HOURS = 24
HORIZONS_HOURS = (24, 72, 168)
MIN_PROBABILITY_ROWS = 24 * 12

conn = connect(DB_PATH)
markets_df = load_multi_domain_markets(
    conn,
    domains=DOMAINS,
    max_markets_per_domain=MAX_MARKETS_PER_DOMAIN,
    min_probability_rows=MIN_PROBABILITY_ROWS,
)
probabilities_df = load_probabilities_for_markets(conn, markets_df["market_id"].tolist())

trust_df = build_multi_horizon_terminal_dataset(
    markets_df,
    probabilities_df,
    horizons_hours=HORIZONS_HOURS,
    max_snapshot_staleness_hours=12,
)
trust_df = add_time_features(trust_df)
trust_df = trust_df.loc[trust_df["horizon_hours"] == TARGET_HORIZON_HOURS].reset_index(drop=True)
market_feature_cols = [
    "primary_domain",
    "market_text",
    "family_id",
    "candidate_family_id",
    "tag_count",
    "matched_domain_count",
    "question_char_len",
    "description_char_len",
    "has_resolution_source",
    "duplicate_neighbor_count",
    "max_text_similarity",
    "family_market_count",
    "family_volume_sum",
    "family_end_date_span_days",
    "candidate_market_count",
    "candidate_volume_sum",
] + [
    col for col in markets_df.columns if col.startswith("kw_") or col.startswith("text_svd_")
]
trust_df = trust_df.merge(markets_df[["market_id", *market_feature_cols]], on="market_id", how="left")
trust_df = attach_family_snapshot_features(
    trust_df,
    probabilities_df,
    markets_df,
    time_col="cutoff_timestamp_utc",
    prob_col="current_yes_probability",
)
covariates_df = load_optional_covariates(EXTERNAL_COVARIATES_PATH)
trust_df = join_covariates(trust_df, covariates_df, time_col="cutoff_timestamp_utc")
trust_df = add_manipulation_proxies(trust_df)
trust_df = add_domain_dummies(trust_df)

panels = {market_id: frame.reset_index(drop=True) for market_id, frame in probabilities_df.groupby("market_id", sort=False)}
future_rows = []
for row in trust_df[["market_id", "cutoff_timestamp_utc", "current_yes_probability"]].itertuples(index=False):
    panel = panels[str(row.market_id)]
    future = panel.loc[panel["timestamp_utc"] >= pd.Timestamp(row.cutoff_timestamp_utc)]
    probs = future["yes_probability"].to_numpy(dtype=float)
    if len(probs) <= 1:
        future_rows.append({
            "market_id": str(row.market_id),
            "cutoff_timestamp_utc": pd.Timestamp(row.cutoff_timestamp_utc),
            "remaining_instability": 0.0,
            "max_future_dislocation": 0.0,
            "terminal_repricing": 0.0,
        })
        continue
    future_rows.append({
        "market_id": str(row.market_id),
        "cutoff_timestamp_utc": pd.Timestamp(row.cutoff_timestamp_utc),
        "remaining_instability": float(np.abs(np.diff(probs)).sum()),
        "max_future_dislocation": float(np.max(np.abs(probs - float(row.current_yes_probability)))),
        "terminal_repricing": float(abs(probs[-1] - float(row.current_yes_probability))),
    })
future_df = pd.DataFrame(future_rows)
trust_df = trust_df.merge(future_df, on=["market_id", "cutoff_timestamp_utc"], how="left")

for col in ["market_abs_error", "market_log_loss", "remaining_instability", "max_future_dislocation", "terminal_repricing", "family_coherence_gap", "candidate_prob_sum_gap"]:
    scale = float(trust_df[col].quantile(0.9)) if trust_df[col].notna().any() else 1.0
    scale = scale if scale > 1e-6 else 1.0
    trust_df[f"scaled_{col}"] = trust_df[col] / scale

trust_df["composite_trust_risk"] = (
    0.40 * trust_df["scaled_market_log_loss"]
    + 0.25 * trust_df["scaled_remaining_instability"]
    + 0.20 * trust_df["scaled_family_coherence_gap"]
    + 0.15 * trust_df["scaled_candidate_prob_sum_gap"]
)

print(f"rows at {TARGET_HORIZON_HOURS}h horizon: {len(trust_df):,}")
display(
    trust_df.groupby("primary_domain", dropna=False)
    .agg(rows=("market_id", "size"), mean_abs_error=("market_abs_error", "mean"), mean_composite_risk=("composite_trust_risk", "mean"))
    .reset_index()
    .sort_values("rows", ascending=False)
)
display(trust_df[["market_abs_error", "remaining_instability", "family_coherence_gap", "candidate_prob_sum_gap", "composite_trust_risk"]].describe().T)


rows at 24h horizon: 262


## Evaluation Protocol and Baselines

**Split:**
- out-of-time evaluation;
- train/test are separated by time to avoid contamination.

**Metrics:**
- mean log loss on the covered subset;
- coverage curves;
- comparison of learned trust against simple heuristic policies.

**Selective prediction:**
- the model may abstain on part of the examples;
- therefore it is important to inspect not only quality, but also the price of that quality in terms of coverage.

**Baselines:**
- confidence margin;
- simple heuristic combinations;
- a learned regressor / scorer that predicts error risk.

In [3]:
feature_family_map = {
    "text": [col for col in trust_df.columns if col.startswith("text_svd_") or col.startswith("kw_") or col in {"tag_count", "matched_domain_count", "question_char_len", "description_char_len", "has_resolution_source", "duplicate_neighbor_count", "max_text_similarity"}],
    "graph": [col for col in trust_df.columns if col.startswith("family_") or col.startswith("candidate_")],
    "external": [col for col in trust_df.columns if col.startswith("btc_usd") or col.startswith("eth_usd")],
    "manipulation": [col for col in trust_df.columns if col.endswith("_proxy")],
    "domain": [col for col in trust_df.columns if col.startswith("domain_")],
}
base_excluded = {
    "market_id", "market_slug", "question", "end_date", "created_at", "cutoff_timestamp_utc", "target", "market_text",
    "family_id", "candidate_family_id", "market_abs_error", "market_log_loss", "resolution_source", "tag_labels", "matched_domains",
    "remaining_instability", "max_future_dislocation", "terminal_repricing", "composite_trust_risk",
}
trust_feature_cols = [
    col for col in trust_df.columns
    if pd.api.types.is_numeric_dtype(trust_df[col])
    and col not in base_excluded
]

coverage_grid = (0.1, 0.2, 0.4, 0.6, 0.8, 1.0)
rows = []
for train_df, test_df, meta in rolling_time_splits(trust_df, time_col="end_date", n_splits=4, min_train_fraction=0.5):
    error_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("reg", HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=300, random_state=42)),
    ])
    risk_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("reg", HistGradientBoostingRegressor(max_depth=5, learning_rate=0.04, max_iter=350, random_state=42)),
    ])
    error_model.fit(train_df[trust_feature_cols], train_df["market_abs_error"])
    risk_model.fit(train_df[trust_feature_cols], train_df["composite_trust_risk"])

    policies = {
        "confidence_margin": test_df["confidence_margin"].to_numpy(dtype=float),
        "margin_x_activity": test_df["confidence_margin"].to_numpy(dtype=float) * (1.0 + test_df["lookback_24h_observed_trade_share"].fillna(0.0).to_numpy(dtype=float)),
        "coherence_aware_heuristic": test_df["confidence_margin"].to_numpy(dtype=float) / (1.0 + test_df["family_coherence_gap"].fillna(0.0).to_numpy(dtype=float) + 0.2 * test_df["wash_proxy"].fillna(0.0).to_numpy(dtype=float)),
        "learned_error_only": -error_model.predict(test_df[trust_feature_cols]),
        "learned_composite_risk": -risk_model.predict(test_df[trust_feature_cols]),
        "oracle": -test_df["composite_trust_risk"].to_numpy(dtype=float),
    }

    for policy_name, score in policies.items():
        ordered = test_df.assign(_score=score).sort_values("_score", ascending=False).reset_index(drop=True)
        for coverage in coverage_grid:
            keep_n = max(1, int(len(ordered) * coverage))
            kept = ordered.iloc[:keep_n]
            rows.append({
                "fold": meta["fold"],
                "policy": policy_name,
                "coverage": coverage,
                "mean_market_log_loss": float(kept["market_log_loss"].mean()),
                "mean_abs_error": float(kept["market_abs_error"].mean()),
                "mean_remaining_instability": float(kept["remaining_instability"].mean()),
                "mean_family_coherence_gap": float(kept["family_coherence_gap"].mean()),
                "mean_composite_risk": float(kept["composite_trust_risk"].mean()),
            })

coverage_df = pd.DataFrame(rows)
display(
    coverage_df.groupby(["policy", "coverage"])[["mean_market_log_loss", "mean_remaining_instability", "mean_family_coherence_gap", "mean_composite_risk"]]
    .mean()
    .reset_index()
    .sort_values(["coverage", "mean_composite_risk"])
)


## Results

In this block, you should not focus on a single number, but on the trade-off:
- how much the error decreases,
- at what coverage,
- and how stable the effect is across folds.

The key question is: **can we learn to keep only those market states where the market is genuinely worth trusting?**

In [4]:
plot_df = (
    coverage_df.groupby(["policy", "coverage"])[["mean_composite_risk", "mean_market_log_loss", "mean_remaining_instability"]]
    .mean()
    .reset_index()
)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.lineplot(data=plot_df, x="coverage", y="mean_composite_risk", hue="policy", marker="o", ax=axes[0])
axes[0].set_title("Selective composite trust risk")
sns.lineplot(data=plot_df, x="coverage", y="mean_market_log_loss", hue="policy", marker="o", ax=axes[1])
axes[1].set_title("Selective terminal log loss")
sns.lineplot(data=plot_df, x="coverage", y="mean_remaining_instability", hue="policy", marker="o", ax=axes[2])
axes[2].set_title("Selective future instability")
for ax in axes:
    ax.legend_.remove()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

display(
    trust_df.groupby(pd.qcut(trust_df["composite_trust_risk"], q=5, duplicates="drop"))[["market_abs_error", "remaining_instability", "family_coherence_gap", "candidate_prob_sum_gap"]]
    .mean()
)


## Interpretation

The main interpretation here is not “we built another scorer.”

A strong story looks like this:
- the learned policy can find states where the market probability is genuinely more reliable;
- selective prediction improves quality without collapsing coverage;
- the trust signal is tied not only to confidence margin, but also to future path instability, coherence pressure, and liquidity regime.

If a simple heuristic baseline remains very strong, that is still useful:
- it means most of the trust signal is already contained in the market probability itself;
- more complex models should only be justified if they provide a stable gain over that simple heuristic.